In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


In [2]:
from src.train import run
report = run()
report['best_model'], report['model_ranking_by_cv_pr_auc']

ModuleNotFoundError: No module named 'mlflow'

## Six-model comparison (5-fold stratified CV, PR-AUC led)

In [3]:
cv_df = pd.DataFrame(report['cv_results']).T[
    ['pr_auc','roc_auc','recall','precision','f1','accuracy']
].sort_values('pr_auc', ascending=False)
cv_df.style.format('{:.4f}').background_gradient(subset=['pr_auc'], cmap='Greens')

NameError: name 'report' is not defined

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cv_df['pr_auc'].plot(kind='bar', ax=ax, color='#3b6ea5')
ax.set_ylabel('CV PR-AUC')
ax.set_title('Model comparison — PR-AUC (imbalance-aware headline metric)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Imbalance strategy benchmark

`config.yaml` pins `imbalance.strategy: class_weight` as the default after benchmarking three approaches — class_weight/scale_pos_weight, SMOTE (inside the pipeline via imblearn, so it's refit per fold and never touches validation rows), and plain threshold tuning. class_weight matched SMOTE's PR-AUC in this benchmark while being cheaper and leak-free by construction (no synthetic sampling step to audit). Set `imbalance.strategy: smote` in config.yaml and re-run this notebook to reproduce the comparison.

## Value-based decision threshold

0.5 is meaningless on an imbalanced problem — the threshold is chosen from `expected_value = tp * value_per_conversion - (tp+fp) * cost_per_intervention`, swept across the PR curve (see `src/utils/metrics.py::find_value_based_threshold`).

In [ ]:
thr = report['chosen_threshold_info']
print(f"Chosen threshold: {thr['threshold']:.3f}")
print(f"Captures {thr['conversions_captured']}/{thr['total_conversions']} conversions "
      f"via {thr['interventions_triggered']} interventions "
      f"(precision={thr['precision_at_threshold']:.3f}, recall={thr['recall_at_threshold']:.3f})")

## Selected model & held-out test performance

In [ ]:
test_m = report['test_metrics_chosen_threshold']
pd.Series({k: v for k, v in test_m.items() if k != 'confusion_matrix'})

In [ ]:
cm = test_m['confusion_matrix']
cm_df = pd.DataFrame([[cm['tn'], cm['fp']], [cm['fn'], cm['tp']]],
                     index=['Actual: No purchase', 'Actual: Purchase'],
                     columns=['Pred: No purchase', 'Pred: Purchase'])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title(f"Confusion matrix — {report['best_model']} @ threshold={thr['threshold']:.3f}")
plt.show()

**Conclusion:** the best model and its fitted preprocessing pipeline are saved to `models/` for the API and dashboard to load directly — training never happens inside serving code.